# Sarsa on Windy Gridworld

Reproduces **S&B Example 6.5** (§6.4). The agent learns the shortest path S → G
under an upward crosswind (reward −1 per step) using on-policy TD control
(**Sarsa**, `ε=0.1, α=0.5, γ=1`). All figures are interactive — hover for values.

In [1]:
import numpy as np

from rl.envs.windy_gridworld import WindyGridworld
from rl.envs.wrappers import EpisodeStats
from rl.td.sarsa import sarsa

from windy_gridworld_plots import (
    plot_gridworld,
    plot_learning_curve,
    plot_steps_per_episode,
    plot_greedy_policy,
    plot_state_values,
    plot_greedy_trajectory,
    plot_policy_evolution,
)

## 1. The environment

Shaded columns are windy — the number is how many cells you get pushed *up*. Hover any cell.

In [2]:
env = WindyGridworld()  # 4 actions, deterministic wind, gamma = 1
plot_gridworld(env).show()

## 2. Train

`EpisodeStats` logs each episode's length; the `on_episode_end` callback snapshots
`Q` after every episode. Neither touches the Sarsa code.

In [3]:
rng = np.random.default_rng(seed=0)
env_stats = EpisodeStats(env)

snapshots = []
def record_Q(episode, Q):
    snapshots.append((episode, Q.copy()))   # copy: Q is updated in place

Q, pi = sarsa(env_stats, rng, n_episodes=500, alpha=0.5, epsilon=0.1, on_episode_end=record_Q)
steps = env_stats.episode_lengths

print(f'episodes run     : {len(steps)}')
print(f'total time steps : {sum(steps):,}')
print(f'first episode    : {steps[0]:,} steps')
print(f'mean of last 50  : {np.mean(steps[-50:]):.1f} steps')

episodes run     : 500
total time steps : 15,141
first episode    : 943 steps
mean of last 50  : 20.2 steps


## 3. Learning curve

Episodes completed vs time steps (the Example 6.5 plot): **steeper = shorter episodes = better**. The second plot is the same data as steps-per-episode.

In [4]:
plot_learning_curve(steps, optimal_steps=15).show()

In [5]:
plot_steps_per_episode(steps, optimal_steps=15, window=10).show()

## 4. Learned policy & values

Greedy action per cell, the value function `V(s) = maxₐ Q(s, a)` (heatmap + 3D surface with the path on it), and the greedy S → G trajectory (optimal = 15 steps).

In [6]:
plot_greedy_policy(env, pi).show()

In [7]:
plot_state_values(env, Q).show()

In [8]:
fig, n_steps = plot_greedy_trajectory(env, pi)
print(f'greedy trajectory length: {n_steps} steps (optimal = 15)')
fig.show()

greedy trajectory length: 15 steps (optimal = 15)


## 5. How it forms (animation)

The greedy policy across training, from the `Q` snapshots. Arrows = best action,
**red** = greedy path, **yellow** = ε-greedy rollout. Press play or drag the slider.

In [9]:
plot_policy_evolution(env, snapshots).show()